# SceneTwin — Proof of Concept
Test whether TRIBE v2 can measure how well a text description preserves a video scene's predicted brain response.

In [ ]:
# 1. Install everything
!git clone https://github.com/facebookresearch/tribev2.git
!pip install -e tribev2 --quiet
!pip install huggingface_hub "numpy==2.2.6" --quiet

# Restart runtime to reload numpy
import os
os.kill(os.getpid(), 9)

In [ ]:
# 2. HuggingFace login
from huggingface_hub import login
login()

In [ ]:
# 3. Download a short test clip (Sintel trailer — same one Meta uses)
!wget -q 'https://download.blender.org/demo/movies/Sintel.2010.720p.mkv' -O /content/sintel.mkv
# Trim to first 30 seconds
!ffmpeg -y -i /content/sintel.mkv -t 30 -c:v libx264 -c:a aac /content/scene.mp4 -loglevel quiet
print('Video ready: /content/scene.mp4')

In [ ]:
# 4. Load TRIBE v2
import sys
sys.path.insert(0, '/content/tribev2')

from tribev2 import TribeModel
model = TribeModel.from_pretrained('facebook/tribev2')
print('Model loaded.')

In [ ]:
# 5. Run TRIBE on the video
import numpy as np

video_events = model.get_events_dataframe(video_path='/content/scene.mp4')
video_preds, _ = model.predict(video_events)
print(f'Video predictions shape: {video_preds.shape}')
np.save('/content/video_preds.npy', video_preds)

In [ ]:
# 6. Write descriptions — good, detailed, bad, hallucinated
descriptions = {
    'accurate_concise': """A young woman with short hair stands in a snowy landscape.
She looks determined and sorrowful. Snow falls around her.
The scene is quiet and cold. She begins walking forward through the snow.""",

    'accurate_detailed': """A young woman with short dark hair stands alone in a vast,
snow-covered landscape under a grey sky. Her expression is a mix of
determination and deep sadness. Large snowflakes drift slowly around her.
She wears layered clothing suitable for harsh winter conditions.
The terrain is flat and featureless, stretching to the horizon.
She takes a breath, then begins walking steadily forward through
the deep snow, leaving footprints behind her.""",

    'bad_vague': """Something happens on screen. A person is there.
The weather looks cold maybe. They move.""",

    'hallucinated': """A man in a red suit stands on a tropical beach.
Palm trees sway in warm wind. He is laughing and holding a drink.
Children play in the ocean behind him. The sun is bright and hot."""
}

# Save each description to a text file
for name, text in descriptions.items():
    path = f'/content/desc_{name}.txt'
    with open(path, 'w') as f:
        f.write(text)
    print(f'Saved {path} ({len(text.split())} words)')

In [ ]:
# 7. Run TRIBE on each description
desc_preds = {}
for name in descriptions:
    print(f'\nRunning TRIBE on: {name}')
    path = f'/content/desc_{name}.txt'
    events = model.get_events_dataframe(text_path=path)
    preds, _ = model.predict(events)
    desc_preds[name] = preds
    np.save(f'/content/desc_{name}_preds.npy', preds)
    print(f'  Shape: {preds.shape}')

print('\nAll descriptions processed.')

In [ ]:
# 8. Compute Scene Preservation Scores
from numpy.linalg import norm

def cosine_sim(a, b):
    return np.dot(a, b) / (norm(a) * norm(b))

# Average across time to get one brain map per stimulus
video_avg = video_preds.mean(axis=0)

print('Scene Preservation Scores (cosine similarity to video):')
print('=' * 55)
scores = {}
for name, preds in desc_preds.items():
    desc_avg = preds.mean(axis=0)
    score = cosine_sim(video_avg, desc_avg)
    scores[name] = score
    word_count = len(descriptions[name].split())
    efficiency = score / word_count
    print(f'  {name:25s}  score: {score:.4f}  words: {word_count:3d}  efficiency: {efficiency:.5f}')

print(f'\nBest description: {max(scores, key=scores.get)}')
print(f'Worst description: {min(scores, key=scores.get)}')

In [ ]:
# 9. Visualize brain maps side by side
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 5, figsize=(25, 4))

# Video
axes[0].imshow(video_avg.reshape(1, -1), aspect='auto', cmap='RdBu_r')
axes[0].set_title(f'Original Video\n(reference)', fontsize=10)
axes[0].set_yticks([])

# Descriptions
for i, (name, preds) in enumerate(desc_preds.items()):
    desc_avg = preds.mean(axis=0)
    axes[i+1].imshow(desc_avg.reshape(1, -1), aspect='auto', cmap='RdBu_r')
    axes[i+1].set_title(f'{name}\nscore: {scores[name]:.4f}', fontsize=10)
    axes[i+1].set_yticks([])

fig.suptitle('SceneTwin: Scene Preservation Scores', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('/content/scenetwin_results.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved to /content/scenetwin_results.png')

In [ ]:
# 10. Bar chart of scores
import matplotlib.pyplot as plt

names = list(scores.keys())
vals = [scores[n] for n in names]
colors = ['#2ecc71', '#27ae60', '#e67e22', '#e74c3c']

fig, ax = plt.subplots(figsize=(8, 5))
bars = ax.bar(range(len(names)), vals, color=colors, edgecolor='white', linewidth=1.5)
ax.set_xticks(range(len(names)))
ax.set_xticklabels([n.replace('_', '\n') for n in names], fontsize=11)
ax.set_ylabel('Scene Preservation Score (cosine similarity)', fontsize=12)
ax.set_title('SceneTwin: How Well Does Each Description Preserve the Scene?', fontsize=13, fontweight='bold')

for bar, val in zip(bars, vals):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
            f'{val:.3f}', ha='center', fontsize=12, fontweight='bold')

ax.set_ylim(0, max(vals) * 1.15)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.savefig('/content/scenetwin_barchart.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved to /content/scenetwin_barchart.png')

In [ ]:
# 11. Download results
from google.colab import files
files.download('/content/scenetwin_results.png')
files.download('/content/scenetwin_barchart.png')
files.download('/content/video_preds.npy')